# 선데이메이플 보상 패턴 분석 · 다음 선데이는 무엇인가

이 노트북은 보관된 원본 로드, 전처리, 시각화, 예측을 한 번에 실행하는 발표용 흐름이다. 재크롤링 없이 `data/raw/` 보관본에서 시작하고, 최종 tidy는 메모리에서만 만든다.

## 1. 데이터 수집

인벤 게시글 1건을 1회 수집해 종류별 뷰와 날짜별 뷰 원시 텍스트로 보관했다. 발표 실행 중에는 네트워크 요청을 하지 않고 보관본만 읽는다.

In [ ]:
from pathlib import Path
import json
import logging
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import compute_cycle_days, merge_is_discontinued
from src.normalize import categorize, extract_magnitude, normalize_reward
from src.parse import parse_by_date, parse_by_type
from src.predict_dataset import DEFAULT_RECENCY_GATE_DAYS, DOMAIN_DISCONTINUED, _candidate_rewards, _eligible_rows, build_inference_frame, build_prediction_frame, gate_candidates
from src.prophet_cycle import build_weekly_series, fit_predict_reward, rank_candidates
from src.viz import set_korean_font

warnings.filterwarnings("ignore", message="IProgress not found.*")
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").disabled = True
logging.getLogger("prophet.plot").disabled = True

TODAY = pd.Timestamp("2026-06-10")
SPLIT_CUTOFF = pd.Timestamp("2025-01-01")
RANDOM_SEED = 42
raw_dir = ROOT / "data" / "raw"
manual_obs_path = ROOT / "data" / "interim" / "manual_obs_20260614.csv"
meta = json.loads((raw_dir / "inven_20563_meta.json").read_text(encoding="utf-8"))
by_type_text = (raw_dir / "view_by_type.txt").read_text(encoding="utf-8")
by_date_text = (raw_dir / "view_by_date.txt").read_text(encoding="utf-8")
source_summary = {
    "출처": meta["url"],
    "수집시각": meta["fetched_at"],
    "종류별_문자수": len(by_type_text),
    "날짜별_문자수": len(by_date_text),
}
display(source_summary)
display(pd.DataFrame({"종류별_앞부분": by_type_text.splitlines()[:8]}))
display(pd.DataFrame({"날짜별_앞부분": by_date_text.splitlines()[:8]}))

## 2. 데이터 전처리

두 원시 뷰를 기존 `src` 파서로 정규식 파싱하고, 보상명 정규화·카테고리 매핑·재등장 주기·단종 여부를 붙여 14컬럼 tidy 데이터를 만든다. 관측 단위는 `(date, reward_raw)`이며, 이 노트북은 결과를 `data/processed`에 저장하지 않는다.

In [ ]:
by_type_records = parse_by_type(by_type_text)
by_date_records = parse_by_date(by_date_text)
parsed_by_type = pd.DataFrame(
    by_type_records,
    columns=["reward_raw", "date", "stated_cycle_days", "is_discontinued", "trailing"],
)
parsed_by_date = pd.DataFrame(
    by_date_records,
    columns=["date", "event_context", "is_major_event", "no_event_week", "reward_raw", "reward_trailing"],
)

def is_special(row):
    text = " ".join(
        str(x)
        for x in [row.get("reward_raw"), row.get("reward_trailing"), row.get("event_context")]
        if pd.notna(x)
    )
    return "타임" in text

def time_limited(trailing):
    if pd.isna(trailing) or trailing is None:
        return None
    text = str(trailing)
    if "하루 종일" in text or "하루종일" in text:
        return "하루 종일"
    if "오전" in text or "오후" in text or "~" in text:
        return text
    if re.search(r"\d+분", text):
        return text
    return None

tidy_prenorm = parsed_by_date.copy()
tidy_prenorm["is_special"] = tidy_prenorm.apply(is_special, axis=1)
tidy_prenorm["time_limited"] = tidy_prenorm["reward_trailing"].apply(time_limited)
tidy_prenorm["is_future"] = pd.to_datetime(tidy_prenorm["date"]) > TODAY
tidy_prenorm = tidy_prenorm[
    [
        "date",
        "reward_raw",
        "reward_trailing",
        "event_context",
        "is_major_event",
        "is_special",
        "time_limited",
        "is_future",
        "no_event_week",
    ]
]

tidy = tidy_prenorm.copy()
tidy["reward_norm"] = tidy["reward_raw"].apply(lambda value: normalize_reward(str(value)) if pd.notna(value) else None)
tidy["category"] = tidy["reward_norm"].apply(lambda value: categorize(value) if pd.notna(value) else None)
tidy["reward_magnitude"] = tidy["reward_raw"].apply(lambda value: extract_magnitude(str(value)) if pd.notna(value) else None)
tidy["cycle_days"] = compute_cycle_days(tidy)
tidy["is_discontinued"] = merge_is_discontinued(tidy, parsed_by_type)
tidy = tidy[
    [
        "date",
        "reward_raw",
        "reward_trailing",
        "reward_norm",
        "category",
        "reward_magnitude",
        "event_context",
        "is_major_event",
        "is_special",
        "time_limited",
        "is_discontinued",
        "is_future",
        "no_event_week",
        "cycle_days",
    ]
]
tidy["date"] = pd.to_datetime(tidy["date"])
analysis = tidy.loc[tidy["reward_norm"].notna() & ~tidy["is_future"].astype(bool)].copy()
display(tidy.head())

In [ ]:
non_sunday_rows = tidy.loc[tidy["date"].dt.weekday != 6, ["date", "reward_norm", "event_context"]].copy()
non_sunday_rows["date"] = non_sunday_rows["date"].dt.strftime("%Y-%m-%d")
quality_summary = {
    "total_rows": len(tidy),
    "analysis_rows": len(analysis),
    "columns": len(tidy.columns),
    "unique_reward_norm": analysis["reward_norm"].nunique(),
    "discontinued_unique_reward_raw": parsed_by_type.loc[parsed_by_type["is_discontinued"], "reward_raw"].nunique(),
    "date_reward_raw_duplicates": int(tidy.loc[tidy["reward_raw"].notna()].duplicated(["date", "reward_raw"]).sum()),
    "non_sunday_rows": len(non_sunday_rows),
}
assert quality_summary["total_rows"] == 1160
assert quality_summary["analysis_rows"] == 1118
assert quality_summary["columns"] == 14
assert quality_summary["discontinued_unique_reward_raw"] == 71
assert quality_summary["date_reward_raw_duplicates"] == 0
assert quality_summary["non_sunday_rows"] == 11
summary_labels = {
    "total_rows": "전체행",
    "analysis_rows": "분석행",
    "columns": "컬럼수",
    "unique_reward_norm": "고유_reward_norm",
    "discontinued_unique_reward_raw": "단종_고유_reward_raw",
    "date_reward_raw_duplicates": "date_reward_raw_중복",
    "non_sunday_rows": "비일요일_행",
}
display(pd.Series(quality_summary).rename(index=summary_labels).to_frame("값"))
display(non_sunday_rows.reset_index(drop=True))

## 3. 시각화

핵심 질문은 네 가지다. 카테고리 구성은 어떻게 변했는가, 한 회차 보상 수는 늘었는가, 재등장 주기는 얼마나 다른가, 단종은 어떤 카테고리에 몰렸는가.

In [ ]:
set_korean_font()
category_year = (
    analysis.assign(연도=analysis["date"].dt.year)
    .groupby(["연도", "category"])
    .size()
    .unstack(fill_value=0)
)
category_share = category_year.div(category_year.sum(axis=1), axis=0)
ax = category_share.plot(kind="bar", stacked=True, figsize=(9, 4))
ax.set_title("연도별 카테고리 비중")
ax.set_xlabel("연도")
ax.set_ylabel("비중")
plt.tight_layout()
plt.show()

session_counts = analysis.groupby("date").size().rename("보상수").reset_index()
session_counts["연도"] = session_counts["date"].dt.year
yearly_session = session_counts.groupby("연도")["보상수"].mean()
ax = yearly_session.plot(kind="line", marker="o", figsize=(8, 4))
ax.set_title("연도별 회차당 평균 보상 수")
ax.set_xlabel("연도")
ax.set_ylabel("평균 보상 수")
plt.tight_layout()
plt.show()

In [ ]:
cycle_values = analysis["cycle_days"].dropna().astype(int)
cycle_capped = cycle_values.clip(upper=365)
ax = cycle_capped.plot(kind="hist", bins=[0, 14, 28, 42, 56, 84, 112, 180, 365], figsize=(8, 4))
ax.set_title("재등장 주기 분포")
ax.set_xlabel("주기일수")
plt.tight_layout()
plt.show()

disc_by_category = (
    analysis.loc[analysis["is_discontinued"].astype("boolean").fillna(False)]
    .drop_duplicates(["reward_raw", "category"])
    .groupby("category")
    .size()
    .sort_values(ascending=False)
)
ax = disc_by_category.plot(kind="bar", figsize=(7, 4))
ax.set_title("카테고리별 단종 고유 보상 수")
ax.set_xlabel("카테고리")
ax.set_ylabel("고유 보상 수")
plt.tight_layout()
plt.show()

## 4. 예측 — 다음 선데이는 무엇인가

분류기는 경과일·과거 출현율·최근 재등장 주기 피처를 사용하고, Prophet은 보상별 주간 0/1 시계열의 28일 주기와 연간 변동을 사용한다. 두 방법은 같은 후보와 같은 `date_hit_rate`로 비교한다. 이 셀은 Prophet 후보 20개를 적합하므로 실행에 수십 초가 걸릴 수 있다.

In [ ]:
prediction_frame = build_prediction_frame(tidy, candidate_cutoff=SPLIT_CUTOFF)
train = prediction_frame.loc[prediction_frame["date"] < SPLIT_CUTOFF].copy()
test = prediction_frame.loc[prediction_frame["date"] >= SPLIT_CUTOFF].copy()
feature_columns = [
    "days_since_last",
    "past_count",
    "past_rate",
    "roll_gap_mean",
    "roll_gap_median",
    "lag1_appeared",
    "category",
    "month",
    "quarter",
    "weekofyear",
]

def evaluate(y_true, y_score, dates):
    y_true = pd.Series(y_true).astype(int).reset_index(drop=True)
    y_score = pd.Series(y_score).astype(float).reset_index(drop=True)
    dates = pd.Series(dates).reset_index(drop=True)
    y_pred = (y_score >= 0.5).astype(int)
    work = pd.DataFrame({"date": dates, "y": y_true, "score": y_score})
    def date_hit(group):
        k = int(group["y"].sum())
        if k < 1:
            return np.nan
        return float(group.nlargest(k, "score")["y"].max())
    auc = roc_auc_score(y_true, y_score) if y_true.nunique() > 1 else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": auc,
        "date_hit_rate": work.groupby("date", sort=True).apply(date_hit).mean(),
    }

X_all = pd.get_dummies(prediction_frame[feature_columns], columns=["category"], dummy_na=True).fillna(-1)
X_train = X_all.loc[train.index]
X_test = X_all.loc[test.index]
y_train = train["y"].astype(int)
y_test = test["y"].astype(int)
b1_score = (test["roll_gap_median"].notna() & (test["days_since_last"] >= test["roll_gap_median"])).astype(float)
b2_score = test["past_rate"].clip(0, 1)
logistic_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
tree_model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED)
logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
logistic_score = logistic_model.predict_proba(X_test)[:, 1]
tree_score = tree_model.predict_proba(X_test)[:, 1]
classifier_results = pd.DataFrame(
    [
        {"방법": "간격규칙", **evaluate(y_test, b1_score, test["date"])},
        {"방법": "사전확률", **evaluate(y_test, b2_score, test["date"])},
        {"방법": "로지스틱", **evaluate(y_test, logistic_score, test["date"])},
        {"방법": "랜덤포레스트", **evaluate(y_test, tree_score, test["date"])},
    ]
)

eligible_for_prophet = _eligible_rows(tidy)
candidates = _candidate_rewards(eligible_for_prophet.loc[eligible_for_prophet["date"] < SPLIT_CUTOFF], 20)
test_dates = pd.Index(sorted(eligible_for_prophet.loc[eligible_for_prophet["date"] >= SPLIT_CUTOFF, "date"].drop_duplicates()))
actual_by_date = {date: set(group["reward_norm"]) for date, group in eligible_for_prophet.groupby("date", sort=False)}
prophet_scores = {}
for reward in candidates:
    series = build_weekly_series(tidy, reward, SPLIT_CUTOFF)
    forecast = fit_predict_reward(series, pd.DatetimeIndex(test_dates))
    prophet_scores[reward] = forecast.set_index("ds")["yhat"]
prophet_hits = []
for date in test_dates:
    gated = gate_candidates(eligible_for_prophet, candidates, date, recency_gate_days=DEFAULT_RECENCY_GATE_DAYS, domain_discontinued=DOMAIN_DISCONTINUED)
    actual = actual_by_date.get(date, set())
    k = len(actual & set(candidates))
    if k < 1 or not gated:
        prophet_hits.append(np.nan)
    else:
        ranked = sorted(gated, key=lambda reward: (-float(prophet_scores[reward].loc[date]), reward))
        prophet_hits.append(float(bool(set(ranked[:k]) & actual)))
prophet_date_hit_rate = pd.Series(prophet_hits).mean()
method_compare = pd.DataFrame(
    [
        {"방법": "간격규칙", "date_hit_rate": float(classifier_results.loc[classifier_results["방법"] == "간격규칙", "date_hit_rate"].iloc[0])},
        {"방법": "사전확률", "date_hit_rate": float(classifier_results.loc[classifier_results["방법"] == "사전확률", "date_hit_rate"].iloc[0])},
        {"방법": "랜덤포레스트", "date_hit_rate": float(classifier_results.loc[classifier_results["방법"] == "랜덤포레스트", "date_hit_rate"].iloc[0])},
        {"방법": "Prophet", "date_hit_rate": prophet_date_hit_rate},
    ]
)
assert round(float(method_compare.loc[method_compare["방법"] == "랜덤포레스트", "date_hit_rate"].iloc[0]), 4) == 0.4493
assert round(float(method_compare.loc[method_compare["방법"] == "Prophet", "date_hit_rate"].iloc[0]), 4) == 0.4203
display(classifier_results)
display(method_compare)

In [ ]:
manual_obs = pd.read_csv(manual_obs_path, parse_dates=["date"])
actual_0614 = set(manual_obs["reward_norm"])

def predict_classifier(tidy_source, target_date, gate):
    infer_frame = build_inference_frame(
        tidy_source,
        target_date,
        candidate_cutoff=SPLIT_CUTOFF,
        recency_gate_days=DEFAULT_RECENCY_GATE_DAYS if gate else None,
        domain_discontinued=DOMAIN_DISCONTINUED if gate else frozenset(),
    )
    X_target = pd.get_dummies(infer_frame[feature_columns], columns=["category"], dummy_na=True)
    X_target = X_target.reindex(columns=X_all.columns, fill_value=0).fillna(-1)
    table = infer_frame[["date", "reward_norm"]].copy()
    table["점수"] = tree_model.predict_proba(X_target)[:, 1]
    table = table.sort_values(["점수", "reward_norm"], ascending=[False, True]).reset_index(drop=True)
    table["순위"] = np.arange(1, len(table) + 1)
    table["점수(%)"] = (table["점수"] * 100).round(1)
    return table

def actual_rank(table, actual_rewards, score_col):
    rows = []
    for reward in sorted(actual_rewards):
        matched = table.loc[table["reward_norm"] == reward]
        if matched.empty:
            rows.append({"reward_norm": reward, "후보포함": False, "순위": pd.NA, score_col: pd.NA})
        else:
            row = matched.iloc[0]
            rows.append({"reward_norm": reward, "후보포함": True, "순위": int(row["순위"]), score_col: float(row[score_col])})
    return pd.DataFrame(rows)

pred_0614_raw = predict_classifier(tidy, pd.Timestamp("2026-06-14"), gate=False)
pred_0614_gate = predict_classifier(tidy, pd.Timestamp("2026-06-14"), gate=True)
gate_summary = pd.DataFrame(
    [
        {"구분": "게이트 전", "상위5_적중수": int(pred_0614_raw.head(5)["reward_norm"].isin(actual_0614).sum()), "후보수": len(pred_0614_raw)},
        {"구분": "게이트 후", "상위5_적중수": int(pred_0614_gate.head(5)["reward_norm"].isin(actual_0614).sum()), "후보수": len(pred_0614_gate)},
    ]
)
rank_raw = actual_rank(pred_0614_raw, actual_0614, "점수(%)").assign(**{"구분": "게이트 전"})
rank_gate = actual_rank(pred_0614_gate, actual_0614, "점수(%)").assign(**{"구분": "게이트 후"})
prophet_0614 = rank_candidates(tidy, candidates, as_of=pd.Timestamp("2026-06-14"), target_date=pd.Timestamp("2026-06-14"), recency_gate_days=DEFAULT_RECENCY_GATE_DAYS, domain_discontinued=DOMAIN_DISCONTINUED)
prophet_0614["yhat"] = prophet_0614["yhat"].round(3)
assert gate_summary.loc[0, "상위5_적중수"] == 0
assert gate_summary.loc[1, "상위5_적중수"] == 1
assert int(rank_raw.loc[rank_raw["reward_norm"] == "몬스터파크_경험치", "순위"].iloc[0]) == 9
assert int(rank_gate.loc[rank_gate["reward_norm"] == "몬스터파크_경험치", "순위"].iloc[0]) == 5
display(gate_summary)
display(pd.concat([rank_raw, rank_gate], ignore_index=True)[["구분", "reward_norm", "후보포함", "순위", "점수(%)"]])
display(prophet_0614.head(5))

In [ ]:
category_lookup = tidy.dropna(subset=["reward_norm", "category"]).groupby("reward_norm")["category"].agg(lambda s: s.mode().iloc[0])
manual_for_tidy = manual_obs.copy()
manual_for_tidy["category"] = manual_for_tidy["reward_norm"].map(category_lookup)
manual_for_tidy["is_future"] = False
manual_for_tidy["no_event_week"] = False
tidy_with_manual = pd.concat([tidy, manual_for_tidy], ignore_index=True, sort=False)
rf_0621 = predict_classifier(tidy_with_manual, pd.Timestamp("2026-06-21"), gate=True)
prophet_0621 = rank_candidates(tidy_with_manual, candidates, as_of=pd.Timestamp("2026-06-21"), target_date=pd.Timestamp("2026-06-21"), recency_gate_days=DEFAULT_RECENCY_GATE_DAYS, domain_discontinued=DOMAIN_DISCONTINUED)
prophet_0621["yhat"] = prophet_0621["yhat"].round(3)
forecast_compare = pd.DataFrame(
    {
        "순위": range(1, 6),
        "랜덤포레스트_보상": rf_0621.head(5)["reward_norm"].to_list(),
        "랜덤포레스트_점수(%)": rf_0621.head(5)["점수(%)"].to_list(),
        "Prophet_보상": prophet_0621.head(5)["reward_norm"].to_list(),
        "Prophet_yhat": prophet_0621.head(5)["yhat"].to_list(),
    }
)
excluded_rewards = {"현상금_경험치", "불꽃늑대_경험치", "스타포스_1+1강화", "스타포스_성공확률"}
assert excluded_rewards.isdisjoint(set(rf_0621["reward_norm"]))
assert excluded_rewards.isdisjoint(set(prophet_0621["reward_norm"]))
assert forecast_compare.loc[0, "랜덤포레스트_보상"] == "몬스터컬렉션"
assert float(forecast_compare.loc[0, "랜덤포레스트_점수(%)"]) == 41.7
assert forecast_compare.loc[0, "Prophet_보상"] == "몬스터파크_경험치"
assert float(forecast_compare.loc[0, "Prophet_yhat"]) == 0.402
display(forecast_compare)

## 5. 결론

선데이메이플은 한 회차에 담기는 보상 수가 늘었고, 최근으로 올수록 강화와 성장 계열이 큰 축을 이룬다. 단종은 성장·사냥효율·수집기타 계열에 집중됐다. 예측에서는 랜덤포레스트가 백테스트 `date_hit_rate` 0.4493으로 Prophet 0.4203보다 높았지만, 두 방법은 서로 다른 신호를 보며 둘 다 단종 게이트가 필요했다.